In [0]:
from azure.eventhub import EventHubProducerClient, EventData
import json

EVENT_HUB_CONNECTION_STRING = dbutils.secrets.get(
scope="key-vault-scope", 
key="eventhub-connection-string")  # From Event Hub policy
EVENT_HUB_NAME = "weatherstreamingehi"  # e.g., weatherstreamingeventhub

producer = EventHubProducerClient.from_connection_string(
    conn_str=EVENT_HUB_CONNECTION_STRING, 
    eventhub_name=EVENT_HUB_NAME
)

def send_event(event):
    event_data_batch = producer.create_batch()
    event_data_batch.add(EventData(json.dumps(event)))
    producer.send_batch(event_data_batch)

event = {"event_id": 2222, "event_name": "Key vault test"}
send_event(event)
producer.close()

In [0]:
import requests
import json

def handle_response(response):
    if response.status_code == 200:
        return response.json()
    else:
        return f"Error: {response.status_code}, {response.text}"

def get_current_weather(base_url, api_key, location):
    url = f"{base_url}/current.json"
    params = {"key": api_key, "q": location, "aqi": "yes"}
    return handle_response(requests.get(url, params=params))

def get_forecast_weather(base_url, api_key, location, days):
    url = f"{base_url}/forecast.json"
    params = {"key": api_key, "q": location, "days": days}
    return handle_response(requests.get(url, params=params))

def get_alerts(base_url, api_key, location):
    url = f"{base_url}/alerts.json"
    params = {"key": api_key, "q": location, "alerts": "yes"}
    return handle_response(requests.get(url, params=params))

def flatten_data(current_weather, forecast_weather, alerts):
    location = current_weather.get("location", {})
    current = current_weather.get("current", {})
    condition = current.get("condition", {})
    air_quality = current.get("air_quality", {})
    forecast = forecast_weather.get("forecast", {}).get("forecastday", [])
    alert_list = alerts.get("alerts", {}).get("alert", [])

    return {
        "name": location.get("name"), "region": location.get("region"), "country": location.get("country"),
        "lat": location.get("lat"), "lon": location.get("lon"), "localtime": location.get("localtime"),
        "temp_c": current.get("temp_c"), "is_day": current.get("is_day"), "condition_text": condition.get("text"),
        "condition_icon": condition.get("icon"), "wind_kph": current.get("wind_kph"), "wind_degree": current.get("wind_degree"),
        "wind_dir": current.get("wind_dir"), "pressure_in": current.get("pressure_in"), "precip_in": current.get("precip_in"),
        "humidity": current.get("humidity"), "cloud": current.get("cloud"), "feelslike_c": current.get("feelslike_c"),
        "uv": current.get("uv"),
        "air_quality": {k: air_quality.get(k) for k in ["co", "no2", "o3", "so2", "pm2_5", "pm10", "us-epa-index", "gb-defra-index"]},
        "alerts": [{"headline": a.get("headline"), "severity": a.get("severity"), "description": a.get("desc"), "instruction": a.get("instruction")} for a in alert_list],
        "forecast": [{"date": d.get("date"), "maxtemp_c": d.get("day", {}).get("maxtemp_c"), "mintemp_c": d.get("day", {}).get("mintemp_c"), "condition": d.get("day", {}).get("condition", {}).get("text")} for d in forecast]
    }

def fetch_weather_data():
    base_url = "http://api.weatherapi.com/v1/"
    location = "Toronto"  # Customize as needed
    weatherapikey = dbutils.secrets.get(scope="key-vault-scope", key="<your-key-vault-secret-name-for-weatherapi-key>")
    current_weather = get_current_weather(base_url, weatherapikey, location)
    forecast_weather = get_forecast_weather(base_url, weatherapikey, location, 3)
    alerts = get_alerts(base_url, weatherapikey, location)
    merged_data = flatten_data(current_weather, forecast_weather, alerts)
    print("Weather Data:", json.dumps(merged_data, indent=3))

fetch_weather_data()

In [0]:
api_key = dbutils.secrets.get(scope="key-vault-scope", key="weatherAPIKey")
eventhub_connections = dbutils.secrets.get(scope="key-vault-scope", key="eventhub-connection-string")
eventhub_name = "weatherstreamingehi"

def fetch_api_data(batch_df, batch_id):
    import requests,json
    from azure.eventhub import EventHubProducerClient, EventData

    location = "Toronto"
    url = "http://api.weatherapi.com/v1/current.json"
    params = {"key": api_key, "q": location,"days":1,"aqi":"yes","alerts":"yes"}
    response = requests.get(url,params = params)
    weather = response.json()
    location = weather.get("location",{})
    current = weather.get("current",{})
    condition = current.get("condition",{})
    air_quality = current.get("air_quality",{})
    forecast = weather.get("forecast",{}).get("forecastday",[])
    alert_list = weather.get("alerts",{}).get("alert",[])
    event =  {
            "name": location.get("name"), "region": location.get("region"), "country": location.get("country"),
            "lat": location.get("lat"), "lon": location.get("lon"), "localtime": location.get("localtime"),
            "temp_c": current.get("temp_c"), "is_day": current.get("is_day"), "condition_text": condition.get("text"),
            "condition_icon": condition.get("icon"), "wind_kph": current.get("wind_kph"), "wind_degree": current.get("wind_degree"),
            "wind_dir": current.get("wind_dir"), "pressure_in": current.get("pressure_in"), "precip_in": current.get("precip_in"),
            "humidity": current.get("humidity"), "cloud": current.get("cloud"), "feelslike_c": current.get("feelslike_c"),
            "uv": current.get("uv"),
            "air_quality": {k: air_quality.get(k) for k in ["co", "no2", "o3", "so2", "pm2_5", "pm10", "us-epa-index", "gb-defra-index"]},
            "alerts": [{"headline": a.get("headline"), "severity": a.get("severity"), "description": a.get("desc"), "instruction": a.get("instruction")} for a in alert_list],
            "forecast": [{"date": d.get("date"), "maxtemp_c": d.get("day", {}).get("maxtemp_c"), "mintemp_c": d.get("day", {}).get("mintemp_c"), "condition": d.get("day", {}).get("condition", {}).get("text")} for d in forecast]
        }

    producer = EventHubProducerClient.from_connection_string(eventhub_connections)
    event_data_batch = producer.create_batch()
    event_data_batch.add(EventData(json.dumps(event)))
    producer.send_batch(event_data_batch)
    producer.close()

    print("Data sent to Event Hub")
# fetch_api_data()
# def fetch_weather_data():
#     location = "Toronto"
#     base_url = "http://api.weatherapi.com/v1"
#     weather = get_weather(base_url,api_key,location,3)
#     location = weather.get("location",{})
#     current = weather.get("current",{})
#     condition = current.get("condition",{})
#     air_quality = current.get("air_quality",{})
#     forecast = weather.get("forecast",{}).get("forecastday",[])
#     alert_list = weather.get("alerts",{}).get("alert",[])
#     json_res =  {
#             "name": location.get("name"), "region": location.get("region"), "country": location.get("country"),
#             "lat": location.get("lat"), "lon": location.get("lon"), "localtime": location.get("localtime"),
#             "temp_c": current.get("temp_c"), "is_day": current.get("is_day"), "condition_text": condition.get("text"),
#             "condition_icon": condition.get("icon"), "wind_kph": current.get("wind_kph"), "wind_degree": current.get("wind_degree"),
#             "wind_dir": current.get("wind_dir"), "pressure_in": current.get("pressure_in"), "precip_in": current.get("precip_in"),
#             "humidity": current.get("humidity"), "cloud": current.get("cloud"), "feelslike_c": current.get("feelslike_c"),
#             "uv": current.get("uv"),
#             "air_quality": {k: air_quality.get(k) for k in ["co", "no2", "o3", "so2", "pm2_5", "pm10", "us-epa-index", "gb-defra-index"]},
#             "alerts": [{"headline": a.get("headline"), "severity": a.get("severity"), "description": a.get("desc"), "instruction": a.get("instruction")} for a in alert_list],
#             "forecast": [{"date": d.get("date"), "maxtemp_c": d.get("day", {}).get("maxtemp_c"), "mintemp_c": d.get("day", {}).get("mintemp_c"), "condition": d.get("day", {}).get("condition", {}).get("text")} for d in forecast]
#         }
#     return json_res

stream = (
    spark.readStream
        .format("rate")
        .option("rowsPerSecond", 1)  # polling interval control
        .load()
)
query = (
    stream.writeStream
        .foreachBatch(fetch_api_data)
        .outputMode("append")
        .trigger(processingTime="30 seconds")
        .start()
)

query.awaitTermination()


In [0]:
import requests
import json
from azure.eventhub import EventHubProducerClient, EventData
from datetime import datetime, timedelta

eventhub_connection_string = dbutils.secrets.get(scope="key-vault-scope", key="eventhub-connection-string")
EVENT_HUB_NAME = "weatherstreamingehi"
weatherapikey = dbutils.secrets.get(scope="key-vault-scope", key="weatherAPIKey")

producer = EventHubProducerClient.from_connection_string(conn_str=eventhub_connection_string, eventhub_name=EVENT_HUB_NAME)

def send_event(event):
    event_data_batch = producer.create_batch()
    event_data_batch.add(EventData(json.dumps(event)))
    producer.send_batch(event_data_batch)

def handle_response(response):
    if response.status_code == 200:
        return response.json()
    else:
        return f"Error: {response.status_code}, {response.text}"

def get_current_weather(base_url, api_key, location):
    url = f"{base_url}/current.json"
    params = {"key": api_key, "q": location, "aqi": "yes"}
    return handle_response(requests.get(url, params=params))

def get_forecast_weather(base_url, api_key, location, days):
    url = f"{base_url}/forecast.json"
    params = {"key": api_key, "q": location, "days": days}
    return handle_response(requests.get(url, params=params))

def get_alerts(base_url, api_key, location):
    url = f"{base_url}/alerts.json"
    params = {"key": api_key, "q": location, "alerts": "yes"}
    return handle_response(requests.get(url, params=params))

def flatten_data(current_weather, forecast_weather, alerts):
    location = current_weather.get("location", {})
    current = current_weather.get("current", {})
    condition = current.get("condition", {})
    air_quality = current.get("air_quality", {})
    forecast = forecast_weather.get("forecast", {}).get("forecastday", [])
    alert_list = alerts.get("alerts", {}).get("alert", [])

    return {
        "name": location.get("name"), "region": location.get("region"), "country": location.get("country"),
        "lat": location.get("lat"), "lon": location.get("lon"), "localtime": location.get("localtime"),
        "temp_c": current.get("temp_c"), "is_day": current.get("is_day"), "condition_text": condition.get("text"),
        "condition_icon": condition.get("icon"), "wind_kph": current.get("wind_kph"), "wind_degree": current.get("wind_degree"),
        "wind_dir": current.get("wind_dir"), "pressure_in": current.get("pressure_in"), "precip_in": current.get("precip_in"),
        "humidity": current.get("humidity"), "cloud": current.get("cloud"), "feelslike_c": current.get("feelslike_c"),
        "uv": current.get("uv"),
        "air_quality": {k: air_quality.get(k) for k in ["co", "no2", "o3", "so2", "pm2_5", "pm10", "us-epa-index", "gb-defra-index"]},
        "alerts": [{"headline": a.get("headline"), "severity": a.get("severity"), "description": a.get("desc"), "instruction": a.get("instruction")} for a in alert_list],
        "forecast": [{"date": d.get("date"), "maxtemp_c": d.get("day", {}).get("maxtemp_c"), "mintemp_c": d.get("day", {}).get("mintemp_c"), "condition": d.get("day", {}).get("condition", {}).get("text")} for d in forecast]
    }

def fetch_weather_data():
    base_url = "http://api.weatherapi.com/v1/"
    location = "Toronto"
    current_weather = get_current_weather(base_url, weatherapikey, location)
    forecast_weather = get_forecast_weather(base_url, weatherapikey, location, 3)
    alerts = get_alerts(base_url, weatherapikey, location)
    return flatten_data(current_weather, forecast_weather, alerts)

last_sent_time = datetime.now() - timedelta(seconds=30)

def process_batch(batch_df, batch_id):
    global last_sent_time
    try:
        current_time = datetime.now()
        if (current_time - last_sent_time).total_seconds() >= 30:
            weather_data = fetch_weather_data()
            send_event(weather_data)
            last_sent_time = current_time
            print(f"Event sent at {last_sent_time}")
    except Exception as e:
        print(f"Error in batch {batch_id}: {str(e)}")
        raise e

streaming_df = spark.readStream.format("rate").option("rowsPerSecond", 1).load()
query = streaming_df.writeStream.foreachBatch(process_batch).start()
query.awaitTermination()
producer.close()